# 02 — Preparación de datos en Colab

Este notebook prepara el dataset para entrenar:
1. Monta Google Drive
2. Clona el repo (o lo copia desde Drive)
3. Instala dependencias
4. Descarga las imágenes desde los CSV (pants y tops)
5. Genera splits estratificados train/val/test

**Asume que ya subiste a Drive:**
- `data/preprocessed/pants_1.csv`
- `data/preprocessed/tops_1.csv`

Estructura sugerida en Drive:
```
/content/drive/MyDrive/master_ia/fashion-extraction/
├── data/preprocessed/
│   ├── pants_1.csv
│   └── tops_1.csv
└── (lo demás se crea automáticamente)
```

## 1. Mount Drive + clonar repo

In [15]:
from google.colab import auth
auth.authenticate_user()
print('Autenticación exitosa.')

Autenticación exitosa.


In [16]:
!pip install gcsfs==2023.10.0

In [17]:
from google.cloud import storage

# Nombre de tu bucket
bucket_name = 'jbj-vision'

try:
    # Inicializamos el cliente de storage
    client = storage.Client()
    bucket = client.bucket(bucket_name)

    # Listamos los primeros 10 archivos para verificar la conexión
    blobs = bucket.list_blobs(max_results=10)

    print(f"Conexión exitosa. Archivos en gs://{bucket_name}:")
    for blob in blobs:
        print(f" - {blob.name}")
except Exception as e:
    print(f"Error al conectar con el bucket: {e}")

Conexión exitosa. Archivos en gs://jbj-vision:
 - config/pipeline_config.yaml
 - data/.DS_Store
 - data/preprocessed/.DS_Store
 - data/preprocessed/pants_1.csv
 - data/preprocessed/tops_1.csv
 - data/processed/.gitkeep
 - data/processed/images_pants.tar.gz
 - data/processed/images_tops.tar.gz
 - data/raw/.gitkeep
 - data/raw/eda_sample/02f0c4dff1afba158bae2718846ee87e.jpg


## 2. Instalar dependencias

In [18]:
!pwd

/content


In [19]:
# 1. Descargamos el archivo desde el bucket al entorno local de Colab
!gcloud storage cp gs://jbj-vision/requirements.txt .

# 2. Instalamos los requirements usando el archivo local
!pip install -q -r requirements.txt

Copying gs://jbj-vision/requirements.txt to file://./requirements.txt


## 3. Setup paths con Drive

In [20]:
import os
from pathlib import Path

# Como no tenemos el archivo YAML ni la carpeta src/ localmente,
# definimos la configuración manualmente aquí apuntando a GCS y a Colab.
GCS_BUCKET = 'gs://jbj-vision'

config = {
    'data': {
        'pants_csv': f'{GCS_BUCKET}/data/preprocessed/pants_1.csv',
        'tops_csv': f'{GCS_BUCKET}/data/preprocessed/tops_1.csv',
        'splits': {
            'stratify_col': 'color_family',
            'train_ratio': 0.8,
            'val_ratio': 0.1,
            'test_ratio': 0.1,
            # BAJAMOS EL MÍNIMO a 3 para evitar que descarte todas las clases en pruebas pequeñas
            'min_samples_per_class': 3
        }
    },
    'paths': {
        'images_pants': '/content/data/images/pants',
        'images_tops': '/content/data/images/tops',
        'splits': '/content/data/splits',
        'models': '/content/models'
    },
    'seed': 42
}

# Crear directorios locales si no existen
for path_key in ['images_pants', 'images_tops', 'splits', 'models']:
    os.makedirs(config['paths'][path_key], exist_ok=True)

print('Configuración actualizada con min_samples_per_class = 3')

Configuración actualizada con min_samples_per_class = 3


## 4. Descargar imágenes (sample estratificado)
Si ya corriste este notebook antes, el cache de imágenes está en GCS como tarball. Lo descargamos para no re-bajar las ~30k imágenes desde internet.

In [21]:
import subprocess, os

def pull_tarball(ds: str):
    """Descarga y extrae el tarball de imágenes desde GCS si existe."""
    gcs_path = f"gs://jbj-vision/data/processed/images_{ds}.tar.gz"
    local_tar = f"/tmp/images_{ds}.tar.gz"
    r = subprocess.run(['gcloud', 'storage', 'ls', gcs_path], capture_output=True)
    if r.returncode == 0:
        print(f'Pulling {gcs_path} ...')
        os.makedirs(config['paths'][f'images_{ds}'], exist_ok=True)
        !gcloud storage cp {gcs_path} {local_tar}
        !tar xzf {local_tar} -C {config['paths'][f'images_{ds}']} --strip-components=1
        print(f'✓ Cache {ds} restaurado.')
        return True
    else:
        print(f'No existe tarball para {ds}, se descargará desde internet.')
        return False

pants_cache_ok = pull_tarball('pants')
tops_cache_ok  = pull_tarball('tops')

Pulling gs://jbj-vision/data/processed/images_pants.tar.gz ...
Copying gs://jbj-vision/data/processed/images_pants.tar.gz to file:///tmp/images_pants.tar.gz
Temporary download file corrupt. Restarting download gs://jbj-vision/data/processed/images_pants.tar.gz

Average throughput: 421.3MiB/s
✓ Cache pants restaurado.
Pulling gs://jbj-vision/data/processed/images_tops.tar.gz ...
Copying gs://jbj-vision/data/processed/images_tops.tar.gz to file:///tmp/images_tops.tar.gz
Temporary download file corrupt. Restarting download gs://jbj-vision/data/processed/images_tops.tar.gz

Average throughput: 179.1MiB/s
✓ Cache tops restaurado.


## 4. Descargar imágenes faltantes

Descarga ~15k imágenes de cada CSV. Idempotente: saltea las que ya están en cache. Si el tarball se restauró arriba, esta celda es muy rápida (solo verifica el cache).

In [22]:
# 1. Listamos el contenido del bucket para encontrar dónde está la carpeta de código
print("Contenido del bucket:")
!gcloud storage ls gs://jbj-vision/

Contenido del bucket:
gs://jbj-vision/requirements.txt
gs://jbj-vision/config/
gs://jbj-vision/data/
gs://jbj-vision/src/


In [23]:
# 1. Copiamos la carpeta src desde tu bucket al entorno local de Colab
!gcloud storage cp -r gs://jbj-vision/src /content/

# 2. Verificamos que se haya descargado correctamente
print("\nContenido de la carpeta local /content/src:")
!ls -l /content/src

Copying gs://jbj-vision/src/__init__.py to file:///content/src/__init__.py
Copying gs://jbj-vision/src/classification/__init__.py to file:///content/src/classification/__init__.py
Copying gs://jbj-vision/src/classification/model.py to file:///content/src/classification/model.py
Copying gs://jbj-vision/src/classification/train.py to file:///content/src/classification/train.py
Copying gs://jbj-vision/src/color/__init__.py to file:///content/src/color/__init__.py
Copying gs://jbj-vision/src/color/color_names.py to file:///content/src/color/color_names.py
Copying gs://jbj-vision/src/color/extractor.py to file:///content/src/color/extractor.py
Copying gs://jbj-vision/src/data/__init__.py to file:///content/src/data/__init__.py
Copying gs://jbj-vision/src/data/colab.py to file:///content/src/data/colab.py
Copying gs://jbj-vision/src/data/csv_dataset.py to file:///content/src/data/csv_dataset.py
Copying gs://jbj-vision/src/data/downloader.py to file:///content/src/data/downloader.py
Copying g

In [24]:
SAMPLE_PANTS = 25000
SAMPLE_TOPS  = 25000

from src.data.downloader import download_csv_images

# Descarga de Pants
log_pants = download_csv_images(
    csv_path=config['data']['pants_csv'],
    cache_dir=config['paths']['images_pants'],
    sample=SAMPLE_PANTS,
    workers=8,
    timeout=15,
    log_path=f"{config['paths']['splits']}/pants_download_log.csv",
)
print(f"\nPants OK: {log_pants['success'].sum()}/{len(log_pants)}")

# Descarga de Tops
log_tops = download_csv_images(
    csv_path=config['data']['tops_csv'],
    cache_dir=config['paths']['images_tops'],
    sample=SAMPLE_TOPS,
    workers=8,
    timeout=15,
    log_path=f"{config['paths']['splits']}/tops_download_log.csv",
)
print(f"\nTops OK: {log_tops['success'].sum()}/{len(log_tops)}")

Descargando pants_1: 100%|██████████| 12800/12800 [1:23:01<00:00,  2.57it/s]



Pants OK: 4692/12800


Descargando tops_1: 100%|██████████| 18558/18558 [1:38:04<00:00,  3.15it/s]



Tops OK: 8979/18558


## 6. Persistir cache en GCS (tarball)
Empaquetamos las imágenes descargadas y las subimos a GCS para no repetir la descarga en sesiones futuras. Solo sube lo que cambió (incremental).

In [25]:
def push_tarball(ds: str):
    cache_dir = config['paths'][f'images_{ds}']
    gcs_path  = f"gs://jbj-vision/data/processed/images_{ds}.tar.gz"
    local_tar = f"/tmp/images_{ds}.tar.gz"
    print(f'Empaquetando {cache_dir} ...')
    !tar czf {local_tar} -C {cache_dir} .
    print(f'Subiendo a {gcs_path} ...')
    !gcloud storage cp {local_tar} {gcs_path}
    print(f'✓ Tarball {ds} subido.')

push_tarball('pants')
push_tarball('tops')

Empaquetando /content/data/images/pants ...
Subiendo a gs://jbj-vision/data/processed/images_pants.tar.gz ...
uploading large objects. If you would like to opt-out and instead
perform a normal upload, run:
`gcloud config set storage/parallel_composite_upload_enabled False`
If you would like to disable this warning, run:
`gcloud config set storage/parallel_composite_upload_enabled True`
Note that with parallel composite uploads, your object might be
uploaded as a composite object
(https://cloud.google.com/storage/docs/composite-objects), which means
that any user who downloads your object will need to use crc32c
checksums to verify data integrity. gcloud storage is capable of
computing crc32c checksums, but this might pose a problem for other
clients.

Copying file:///tmp/images_pants.tar.gz to gs://jbj-vision/data/processed/images_pants.tar.gz

Average throughput: 394.6MiB/s
✓ Tarball pants subido.
Empaquetando /content/data/images/tops ...
Subiendo a gs://jbj-vision/data/processed/ima

## 7. Splits estratificados (filtrando por imágenes descargadas)

In [26]:
from src.data.splits import generate_splits

# Pants
pants_urls = set(log_pants[log_pants['success']]['url'])
pants_paths = generate_splits(
    csv_path=config['data']['pants_csv'],
    output_dir=config['paths']['splits'],
    stratify_col=config['data']['splits']['stratify_col'],
    train_ratio=config['data']['splits']['train_ratio'],
    val_ratio=config['data']['splits']['val_ratio'],
    test_ratio=config['data']['splits']['test_ratio'],
    min_samples_per_class=config['data']['splits']['min_samples_per_class'],
    seed=config['seed'],
    filter_urls=pants_urls,
)
print('Splits pants:', pants_paths)

Splits pants: {'train': PosixPath('/content/data/splits/pants_1_train.csv'), 'val': PosixPath('/content/data/splits/pants_1_val.csv'), 'test': PosixPath('/content/data/splits/pants_1_test.csv')}


In [27]:
# Tops
tops_urls = set(log_tops[log_tops['success']]['url'])
tops_paths = generate_splits(
    csv_path=config['data']['tops_csv'],
    output_dir=config['paths']['splits'],
    stratify_col=config['data']['splits']['stratify_col'],
    train_ratio=config['data']['splits']['train_ratio'],
    val_ratio=config['data']['splits']['val_ratio'],
    test_ratio=config['data']['splits']['test_ratio'],
    min_samples_per_class=config['data']['splits']['min_samples_per_class'],
    seed=config['seed'],
    filter_urls=tops_urls,
)
print('Splits tops:', tops_paths)

Splits tops: {'train': PosixPath('/content/data/splits/tops_1_train.csv'), 'val': PosixPath('/content/data/splits/tops_1_val.csv'), 'test': PosixPath('/content/data/splits/tops_1_test.csv')}


## 8. Sanity check final

In [28]:
import pandas as pd

BUCKET_SPLITS = config['paths']['splits']
print('=== Resumen ===')
for ds in ['pants_1', 'tops_1']:
    for split in ['train', 'val', 'test']:
        p = f"{BUCKET_SPLITS}/{ds}_{split}.csv"
        try:
            df = pd.read_csv(p)
            print(f'  {ds}/{split}: {len(df):,} filas')
        except Exception as e:
            print(f'  {ds}/{split}: ERROR — {e}')

print('\n✓ Listo para entrenar. Ejecuta 03_train_pants_colab.ipynb')

=== Resumen ===
  pants_1/train: 3,900 filas
  pants_1/val: 488 filas
  pants_1/test: 488 filas
  tops_1/train: 7,275 filas
  tops_1/val: 910 filas
  tops_1/test: 910 filas

✓ Listo para entrenar. Ejecuta 03_train_pants_colab.ipynb
